# 02 · Prepare a raster (2020 colour → model-ready grayscale)
Run once per county raster. Output goes to `data/interim/`.

In [ ]:
# --- SETUP (same in every notebook) -----------------------------------
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, subprocess
REPO = "/content/pei-landuse"
GITHUB = "github.com/YOUR_USER/pei-landuse.git"     # private repo
token = userdata.get("GITHUB_TOKEN")                 # Colab ▸ Secrets (key icon). Never paste tokens in cells.
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://{token}@{GITHUB}", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{REPO}/src")
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
CONFIG = "configs/prepare_2020.yaml"

from pei.config import load_config, make_run_dir, log
from pei.preprocess import prepare_raster
import rasterio

cfg = load_config(CONFIG)
run_dir = make_run_dir(cfg)
p = cfg["prepare"]
for f in [p["source"], p.get("reference")]:
    if f:
        with rasterio.open(f) as s:
            log(run_dir, f"{f}: {s.width}x{s.height}, {s.count} band(s), {s.dtypes[0]}, res={s.res}, crs={s.crs}")

Check the printout: set `target_res` in the config to the 1968 pixel size if they differ, then run:

In [ ]:
prepare_raster(p["source"], p["output"], target_res=p.get("target_res"),
               reference_path=p.get("reference"), log=lambda m: log(run_dir, m))